In [1]:
from langgraph.graph import StateGraph,  START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

e:\Lang_Chain\langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

In [4]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str


In [5]:
def generate_joke(state:JokeState):
    prompt=f"Generate a joke on give topic {state['topic']}"
    response = llm.invoke(prompt).content
    return {"joke":response}

In [6]:
def generate_explanation(state:JokeState):
    prompt=f"Generate explanation of given joke {state['joke']} on the topic {state['topic']}"
    response = llm.invoke(prompt).content
    return {"explanation":response}

In [7]:
graph = StateGraph(JokeState)
graph.add_node("Generate_Joke", generate_joke)
graph.add_node("Generate_Explanation", generate_explanation)

graph.add_edge(START, "Generate_Joke")
graph.add_edge("Generate_Joke", "Generate_Explanation")
graph.add_edge("Generate_Explanation",END)

Create Checkpointer

In [8]:
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

create thread 

In [9]:
config1 = {"configurable":{"thread_id":"01"}}

Get State

In [12]:
workflow.invoke({'topic':'Pizza'}, config=config1)

{'topic': 'Pizza',
 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!",
 'explanation': 'The joke!\n\nHere\'s a breakdown of the joke:\n\n**The Setup:** "Why was the pizza in a bad mood?" - This line sets up the expectation that something funny or unexpected is about to happen with a pizza, and that it\'s going to be related to its emotional state (being in a bad mood).\n\n**The Punchline:** "Because it was feeling crusty!" - This line delivers the humor. The word "crusty" has a double meaning here:\n\n1. **Literal meaning:** In the context of pizza, "crusty" refers to the crust of the pizza, which is the outer layer of the dough.\n2. **Idiomatic meaning:** In everyday language, "feeling crusty" is a play on words, as "crusty" can also mean being grumpy, irritable, or having a bad temper.\n\n**The Joke\'s Humor:** The joke relies on this wordplay to create humor. The listener expects a complex reason for the

In [13]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Pizza', 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!", 'explanation': 'The joke!\n\nHere\'s a breakdown of the joke:\n\n**The Setup:** "Why was the pizza in a bad mood?" - This line sets up the expectation that something funny or unexpected is about to happen with a pizza, and that it\'s going to be related to its emotional state (being in a bad mood).\n\n**The Punchline:** "Because it was feeling crusty!" - This line delivers the humor. The word "crusty" has a double meaning here:\n\n1. **Literal meaning:** In the context of pizza, "crusty" refers to the crust of the pizza, which is the outer layer of the dough.\n2. **Idiomatic meaning:** In everyday language, "feeling crusty" is a play on words, as "crusty" can also mean being grumpy, irritable, or having a bad temper.\n\n**The Joke\'s Humor:** The joke relies on this wordplay to create humor. The listener expects a com

In [14]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Pizza', 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!", 'explanation': 'The joke!\n\nHere\'s a breakdown of the joke:\n\n**The Setup:** "Why was the pizza in a bad mood?" - This line sets up the expectation that something funny or unexpected is about to happen with a pizza, and that it\'s going to be related to its emotional state (being in a bad mood).\n\n**The Punchline:** "Because it was feeling crusty!" - This line delivers the humor. The word "crusty" has a double meaning here:\n\n1. **Literal meaning:** In the context of pizza, "crusty" refers to the crust of the pizza, which is the outer layer of the dough.\n2. **Idiomatic meaning:** In everyday language, "feeling crusty" is a play on words, as "crusty" can also mean being grumpy, irritable, or having a bad temper.\n\n**The Joke\'s Humor:** The joke relies on this wordplay to create humor. The listener expects a co

In [20]:
config2={"configurable":{"thread_id":"02"}}

In [21]:
workflow.invoke({'topic':'Samosa'},config=config2)

{'topic': 'Samosa',
 'joke': "Here's a joke for you:\n\nWhy did the samosa go to therapy?\n\nBecause it was feeling crumby on the inside! (get it? like the crispy outside and the savory filling on the inside!)\n\nHope that made you laugh!",
 'explanation': 'Here\'s a breakdown of the joke:\n\n**The Setup:** The joke starts by asking why a samosa (a type of fried or baked pastry filled with spiced potatoes and peas) went to therapy. This establishes the unexpected scenario of a food item seeking therapy, which piques the listener\'s curiosity.\n\n**The Punchline:** The joke reveals that the samosa went to therapy because it was feeling "crumby on the inside." This phrase has a double meaning:\n\n* In a literal sense, a samosa has a crispy exterior (which could be described as "crumby" in texture) and a soft, savory filling on the inside. So, the phrase "crumby on the inside" is a play on words, referencing the samosa\'s internal composition.\n* In an idiomatic sense, "feeling crumby" is

In [22]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Samosa', 'joke': "Here's a joke for you:\n\nWhy did the samosa go to therapy?\n\nBecause it was feeling crumby on the inside! (get it? like the crispy outside and the savory filling on the inside!)\n\nHope that made you laugh!", 'explanation': 'Here\'s a breakdown of the joke:\n\n**The Setup:** The joke starts by asking why a samosa (a type of fried or baked pastry filled with spiced potatoes and peas) went to therapy. This establishes the unexpected scenario of a food item seeking therapy, which piques the listener\'s curiosity.\n\n**The Punchline:** The joke reveals that the samosa went to therapy because it was feeling "crumby on the inside." This phrase has a double meaning:\n\n* In a literal sense, a samosa has a crispy exterior (which could be described as "crumby" in texture) and a soft, savory filling on the inside. So, the phrase "crumby on the inside" is a play on words, referencing the samosa\'s internal composition.\n* In an idiomatic sense, 

In [23]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Samosa', 'joke': "Here's a joke for you:\n\nWhy did the samosa go to therapy?\n\nBecause it was feeling crumby on the inside! (get it? like the crispy outside and the savory filling on the inside!)\n\nHope that made you laugh!", 'explanation': 'Here\'s a breakdown of the joke:\n\n**The Setup:** The joke starts by asking why a samosa (a type of fried or baked pastry filled with spiced potatoes and peas) went to therapy. This establishes the unexpected scenario of a food item seeking therapy, which piques the listener\'s curiosity.\n\n**The Punchline:** The joke reveals that the samosa went to therapy because it was feeling "crumby on the inside." This phrase has a double meaning:\n\n* In a literal sense, a samosa has a crispy exterior (which could be described as "crumby" in texture) and a soft, savory filling on the inside. So, the phrase "crumby on the inside" is a play on words, referencing the samosa\'s internal composition.\n* In an idiomatic sense,

Time Travel

In [24]:
#Fetch state at any checkpoint

workflow.get_state({"configurable":{"thread_id":"01","checkpoint_id":"1f17b5e6-58f0-6084-8002-22a2776f3039"}})

StateSnapshot(values={'topic': 'Pizza'}, next=('Generate_Joke',), config={'configurable': {'thread_id': '01', 'checkpoint_id': '1f17b5e6-58f0-6084-8002-22a2776f3039'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-09T06:21:18.620890+00:00', parent_config={'configurable': {'thread_id': '01', 'checkpoint_ns': '', 'checkpoint_id': '1f17b5e6-58ed-6637-8001-4d7eb1884778'}}, tasks=(PregelTask(id='78e78954-f1cb-154c-8944-1d20e866349b', name='Generate_Joke', path=('__pregel_pull', 'Generate_Joke'), error=None, interrupts=(), state=None, result={'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!"}),), interrupts=())

In [25]:
#fetching steps ahead of any checkpoint
workflow.invoke(None,({"configurable":{"thread_id":"01","checkpoint_id":"1f17b5e6-58f0-6084-8002-22a2776f3039"}}))

{'topic': 'Pizza',
 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!",
 'explanation': 'A classic play on words!\n\nHere\'s a breakdown of the joke:\n\n**Setup:** The joke starts by asking why a pizza was in a bad mood. This sets up the expectation that there\'s something wrong with the pizza, and we need to find out what\'s bothering it.\n\n**Punchline:** The answer is that the pizza was "feeling crusty." Now, here\'s where the wordplay comes in: "crusty" has a double meaning here.\n\n* In the context of pizza, "crusty" refers to the crust of the pizza, which is the outer layer of dough that forms the base of the pizza.\n* However, "crusty" can also mean being grumpy or irritable, like someone who\'s in a bad mood.\n\n**The joke:** The joke relies on this pun, where the word "crusty" is used to connect the pizza\'s physical characteristic (its crust) to its emotional state (being in a bad mood). It\'s a li

In [26]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Pizza', 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!", 'explanation': 'A classic play on words!\n\nHere\'s a breakdown of the joke:\n\n**Setup:** The joke starts by asking why a pizza was in a bad mood. This sets up the expectation that there\'s something wrong with the pizza, and we need to find out what\'s bothering it.\n\n**Punchline:** The answer is that the pizza was "feeling crusty." Now, here\'s where the wordplay comes in: "crusty" has a double meaning here.\n\n* In the context of pizza, "crusty" refers to the crust of the pizza, which is the outer layer of dough that forms the base of the pizza.\n* However, "crusty" can also mean being grumpy or irritable, like someone who\'s in a bad mood.\n\n**The joke:** The joke relies on this pun, where the word "crusty" is used to connect the pizza\'s physical characteristic (its crust) to its emotional state (being in a b

Update State

In [27]:
workflow.update_state({"configurable":{"thread_id":"01","checkpoint_id":"1f17b606-537c-6a42-8004-e42311f3f9a7", "checkpoint_ns":""}},{"topic":"pasta"})

{'configurable': {'thread_id': '01',
  'checkpoint_ns': '',
  'checkpoint_id': '1f17b60d-f342-6174-8005-2efdf84dc40f'}}

In [28]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pasta', 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!"}, next=('Generate_Explanation',), config={'configurable': {'thread_id': '01', 'checkpoint_ns': '', 'checkpoint_id': '1f17b60d-f342-6174-8005-2efdf84dc40f'}}, metadata={'source': 'update', 'step': 5, 'parents': {}}, created_at='2026-07-09T06:39:01.700850+00:00', parent_config={'configurable': {'thread_id': '01', 'checkpoint_ns': '', 'checkpoint_id': '1f17b606-537c-6a42-8004-e42311f3f9a7'}}, tasks=(PregelTask(id='f45612cb-db56-f787-bdea-f49d4d56e775', name='Generate_Explanation', path=('__pregel_pull', 'Generate_Explanation'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Pizza', 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!", 'explanation': 'A classic play on words!\n\nHere\'s a b

In [29]:
workflow.invoke(None,({"configurable":{"thread_id":"01","checkpoint_id":"1f17b606-537c-6a42-8004-e42311f3f9a7"}}))

{'topic': 'Pizza',
 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!",
 'explanation': 'Here\'s a breakdown of the joke:\n\n**The Setup:** The joke starts by asking why a pizza was in a bad mood. This sets up the expectation that there\'s something wrong with the pizza, and we\'re curious to know what it is.\n\n**The Punchline:** The answer is that the pizza was "feeling crusty." This is a play on words, as "crusty" has a double meaning here:\n\n* In the context of pizza, "crusty" refers to the crust of the pizza, which is the outer layer of the dough.\n* In everyday language, "crusty" can also mean grumpy or irritable, implying that someone is in a bad mood.\n\n**The Joke:** The joke relies on this wordplay to create a pun. The speaker is making a clever connection between the pizza\'s physical characteristic (its crust) and its supposed emotional state (being grumpy). The humor comes from the unexpected t

In [30]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Pizza', 'joke': "Here's a pizza joke:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!", 'explanation': 'Here\'s a breakdown of the joke:\n\n**The Setup:** The joke starts by asking why a pizza was in a bad mood. This sets up the expectation that there\'s something wrong with the pizza, and we\'re curious to know what it is.\n\n**The Punchline:** The answer is that the pizza was "feeling crusty." This is a play on words, as "crusty" has a double meaning here:\n\n* In the context of pizza, "crusty" refers to the crust of the pizza, which is the outer layer of the dough.\n* In everyday language, "crusty" can also mean grumpy or irritable, implying that someone is in a bad mood.\n\n**The Joke:** The joke relies on this wordplay to create a pun. The speaker is making a clever connection between the pizza\'s physical characteristic (its crust) and its supposed emotional state (being grumpy). The humor comes f